# Module 8 Notebook: ML Plan

Before building a model, define the prediction task clearly. A model is only as useful as the question you ask it to learn.

In [ ]:
import pandas as pd

try:
    from sklearn.model_selection import train_test_split
    sklearn_available = True
except ModuleNotFoundError:
    sklearn_available = False
    print('scikit-learn is not available here; using a small classroom train/test split fallback.')

    def train_test_split(X, y, test_size=0.25, random_state=42, stratify=None):
        data = X.copy()
        data['_target'] = y.values
        if stratify is not None:
            test_parts = []
            train_parts = []
            for _, group in data.groupby('_target', group_keys=False):
                shuffled = group.sample(frac=1, random_state=random_state)
                n_test = max(1, round(len(shuffled) * test_size))
                test_parts.append(shuffled.iloc[:n_test])
                train_parts.append(shuffled.iloc[n_test:])
            test = pd.concat(test_parts).sample(frac=1, random_state=random_state)
            train = pd.concat(train_parts).sample(frac=1, random_state=random_state)
        else:
            shuffled = data.sample(frac=1, random_state=random_state)
            n_test = round(len(shuffled) * test_size)
            test = shuffled.iloc[:n_test]
            train = shuffled.iloc[n_test:]
        return train.drop(columns='_target'), test.drop(columns='_target'), train['_target'], test['_target']

df = pd.read_csv('phase2_study_support_clean.csv')
df.head()

## 1. Define the target

Our target is `needs_support`. The model will try to predict whether a fictional learner might need extra support.

In [ ]:
target = 'needs_support'
features = ['practice_quiz_avg','weekly_study_hours','sleep_hours','missing_assignments','screen_time_hours','attends_help_session']

X = df[features].copy()
y = df[target].copy()
X.head()

## 1A. Choose the kind of algorithm carefully

An algorithm is the learning method. You do not need to master every algorithm yet, but you should know that different algorithms learn in different ways.

- **Decision tree:** asks split-style questions, like a flowchart.
- **Logistic regression:** finds weighted patterns for yes/no classification.
- **k-nearest neighbors:** compares a new case to similar past cases.

For this beginner project, a decision tree is a good first choice because it is easy to explain.

In [ ]:
algorithm_choice = {
    'task': 'classification',
    'starter_algorithm': 'DecisionTreeClassifier',
    'why_this_one': 'It is beginner-friendly and easier to explain than many black-box models.'
}
algorithm_choice

## 2. Convert categories into numbers

Most starter ML models need numeric input.

In [ ]:
X['attends_help_session'] = X['attends_help_session'].map({'yes': 1, 'no': 0})
y = y.map({'yes': 1, 'no': 0})
X.head()

## 3. Create train and test sets

The model learns from training data. It is judged on testing data it did not train on.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))
print('Target balance in full data:')
print(y.value_counts(normalize=True).round(2))

## 3A. Where validation fits

Professional teams often use three sets: train, validation, and test. The validation set helps tune choices before the final test. To keep this first project manageable, we will use train and test only.

The habit still matters: do not keep changing the model after peeking at the final test score.

## 4. Set a simple baseline

A model should beat a simple guess. If most examples are `no`, always guessing `no` is the baseline to beat.

In [ ]:
baseline_guess = y_train.mode()[0]
baseline_accuracy = (y_test == baseline_guess).mean()
print('Baseline guess:', baseline_guess)
print('Baseline accuracy:', round(baseline_accuracy, 3))

## 5. Risk check

This is a fictional learning-support example. In the real world, a model like this should never be used to label students without human review, context, and privacy protections.

In [ ]:
plan = {
    'target': target,
    'features': features,
    'model_type': 'classification',
    'baseline_to_beat': round(float(baseline_accuracy), 3),
    'human_review_needed': True
}
plan

## Portfolio note

Save your target, features, train/test split, baseline, and one responsible-use warning.

## 6. Transfer the same planning move to other domains

The notebook uses a synthetic support dataset, but the planning logic also applies to delivery, email, playlists, customer renewal, and sports examples.

In [ ]:
domain_transfer = pd.DataFrame([
    {'domain': 'Email', 'target': 'spam_or_not', 'possible_features': 'sender domain, subject words, link count', 'one_limit': 'new scams may use words the model has never seen'},
    {'domain': 'Delivery', 'target': 'late_or_on_time', 'possible_features': 'distance, weather, pickup time, traffic level', 'one_limit': 'holiday traffic may not match normal weeks'},
    {'domain': 'Playlist', 'target': 'skip_or_replay', 'possible_features': 'tempo, genre, artist familiarity, previous skips', 'one_limit': 'taste changes over time'},
    {'domain': 'Customer renewal', 'target': 'renew_or_cancel', 'possible_features': 'usage, support tickets, plan age', 'one_limit': 'one season of data may not generalize'},
])
domain_transfer

## 7. Feature timing and leakage check

A feature is only fair game if it would be available before the prediction and does not already reveal the answer.

In [ ]:
feature_review = pd.DataFrame({
    'feature': features + ['project_score', 'student_code'],
    'available_before_prediction': [True, True, True, True, True, True, False, True],
    'leaks_answer_or_identifier': [False, False, False, False, False, False, True, True],
    'use_in_first_model': [True, True, True, True, True, True, False, False],
})
feature_review

## 8. Write a model plan card

Before training, summarize what the model is and is not allowed to claim.

In [ ]:
model_plan_card = {
    'prediction_question': 'Can the model predict the synthetic needs_support label from available practice signals?',
    'target': target,
    'features': features,
    'split': '75% train / 25% test with stratified labels',
    'baseline': f"always predict {baseline_guess}",
    'success_condition': 'model should beat the baseline and have explainable mistakes',
    'not_allowed_claim': 'This does not identify real learners or make real support decisions.',
}
model_plan_card